# Desafio final: análise de resenhas do JetGPT

A ideia aqui é ler um arquivo de texto cheio de resenhas em idiomas variados, mandar elas pra um modelo de IA rodando localmente, e fechar com uma análise dos sentimentos.

O fluxo tá dividido em etapas. Cada uma tem uma explicação curta antes do código pra você entender o porquê, não só o como.

Pré-requisito: o LM Studio (ou outro servidor compatível) precisa estar rodando antes de chegar na Etapa 2.

## Configuração

Tudo que você pode precisar trocar tá nesta célula. Se mudar de modelo ou de porta, é só aqui.

In [13]:
# Endereço do servidor local — LM Studio usa 1234, Ollama usa 11434
BASE_URL = "http://127.0.0.1:1234/v1"

# Nome do modelo carregado. No LM Studio o nome quase não importa,
# ele usa o que estiver carregado. Mas é bom deixar descritivo.
MODEL_NAME = "qwen2.5"

# Caminho do arquivo de resenhas
ARQUIVO_RESENHAS = "resenhas.txt"

# Onde guardar o cache (pra não reprocessar resenhas já analisadas)
ARQUIVO_CACHE = "cache_resenhas.json"

print(f"Servidor: {BASE_URL}")
print(f"Modelo:   {MODEL_NAME}")
print(f"Arquivo:  {ARQUIVO_RESENHAS}")

Servidor: http://127.0.0.1:1234/v1
Modelo:   qwen2.5
Arquivo:  resenhas.txt


## Dependências

A gente usa a biblioteca `openai`. Pode parecer estranho usar ela com um modelo local, mas tanto o LM Studio quanto o Ollama falam essa mesma língua, então funciona sem precisar reinventar nada. O `matplotlib` entra na Etapa 5 pros gráficos.

In [14]:
# Descomente e rode uma vez se ainda não tiver:
%pip install openai matplotlib

import json
import re
import os
import hashlib
from collections import Counter, defaultdict
from openai import OpenAI
import matplotlib.pyplot as plt

client = OpenAI(base_url=BASE_URL, api_key="not-needed")

print("Cliente pronto.")

Note: you may need to restart the kernel to use updated packages.
Cliente pronto.


## Etapa 1 — Carregar o arquivo

Abre o `.txt`, pega cada linha como um item de lista. O `encoding="utf-8"` é obrigatório aqui porque o arquivo tem acento, ç, caracteres turcos, poloneses, árabes, japoneses. Se esquecer isso, o Python quebra logo na segunda linha.

O `.strip()` em cada linha tira o `\n` do final e qualquer espaço sobrando. O filtro `if linha.strip()` ignora linhas em branco que às vezes aparecem no final do arquivo.

In [15]:
def carregar_resenhas(caminho: str) -> list[str]:
    with open(caminho, "r", encoding="utf-8") as arquivo:
        return [linha.strip() for linha in arquivo if linha.strip()]


resenhas = carregar_resenhas(ARQUIVO_RESENHAS)

print(f"Carreguei {len(resenhas)} resenhas.\n")
print("Amostra:")
for i, r in enumerate(resenhas[:3]):
    print(f"  [{i}] {r[:90]}{'...' if len(r) > 90 else ''}")

Carreguei 25 resenhas.

Amostra:
  [0] user_001|MarieDupont|reclamacao|L'application plante constamment et c'est très frustrant, ...
  [1] user_002|JohnSmith|elogio|JetGPT is absolutely amazing! It helped me write my entire thesi...
  [2] user_003|SarahJones|reclamacao|The latest update broke voice input completely. I cannot us...


## Etapa 2 — Conversar com o modelo

Aqui montamos a função que pega uma resenha, manda pro LM Studio e recebe um JSON estruturado de volta. O prompt é bem firme sobre o formato porque modelo local às vezes resolve filosofar antes de responder.

Pedimos 5 campos no JSON: usuário, resenha original, idioma, tradução pro português, e a avaliação (positiva / negativa / neutra). O idioma entra agora porque vai ser útil na Etapa 6.

A função `extrair_json_da_resposta` tem três tentativas: primeiro tenta parsear direto, depois procura dentro de bloco markdown ` ```json ... ``` `, e por último pega qualquer coisa entre chaves. Isso resolve uns 95% dos casos de modelo teimoso.

In [16]:
SYSTEM_PROMPT = """Você analisa resenhas de aplicativos e devolve dados estruturados.

Para cada resenha recebida, retorne UM objeto JSON com exatamente estes campos:
- "usuario": nome do usuário que escreveu a resenha
- "resenha_original": texto completo no idioma original
- "idioma": idioma da resenha em português (ex: "francês", "inglês", "espanhol", "turco", "polonês", "romeno", "italiano", "sueco", "alemão", "japonês", "árabe", "russo", "grego", "vietnamita", "português")
- "traducao_pt": tradução fiel para o português brasileiro
- "avaliacao": exatamente uma destas três palavras em minúsculo: positiva, negativa, neutra

Retorne SOMENTE o JSON, sem texto antes ou depois, sem markdown, sem ```json.
Use aspas duplas."""


def extrair_json_da_resposta(texto: str) -> dict | None:
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        pass

    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return None


def analisar_resenha(linha: str) -> dict | None:
    try:
        resposta = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Analise esta resenha:\n\n{linha}"},
            ],
            temperature=0.2,
            response_format={"type": "json_object"},
        )
        return extrair_json_da_resposta(resposta.choices[0].message.content)

    except Exception:
        # Modelo pode não suportar response_format — tenta sem
        try:
            resposta = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Analise esta resenha:\n\n{linha}"},
                ],
                temperature=0.2,
            )
            return extrair_json_da_resposta(resposta.choices[0].message.content)
        except Exception as e:
            print(f"Falhou em: {linha[:60]}... -> {e}")
            return None


# Teste rápido com a primeira resenha
print("Testando com a primeira resenha...\n")
teste = analisar_resenha(resenhas[0])
if teste:
    print(json.dumps(teste, ensure_ascii=False, indent=2))
else:
    print("❌ Não rolou. Confere se o servidor tá no ar.")

Testando com a primeira resenha...

{
  "usuario": "MarieDupont",
  "resenha_original": "L'application plante constamment et c'est très frustrant, je ne peux jamais terminer une conversation complète.",
  "idioma": "francês",
  "traducao_pt": "O aplicativo trava constantemente e isso é muito frustrante, eu nunca consigo completar uma conversa inteira.",
  "avaliacao": "negativa"
}


## Etapa 3 — Processar todas as resenhas (com cache)

Aqui processamos as 25 resenhas em loop. A graça dessa etapa é o **cache**: a gente guarda os resultados num arquivo JSON usando o hash MD5 da resenha como chave. Se você rodar o notebook de novo, ele lê do cache e pula a parte demorada de chamar o modelo.

Útil porque modelo local não é exatamente um foguete — uns 30 segundos por resenha em média, dependendo da máquina.

In [17]:
def carregar_cache() -> dict:
    if os.path.exists(ARQUIVO_CACHE):
        with open(ARQUIVO_CACHE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def salvar_cache(cache: dict) -> None:
    with open(ARQUIVO_CACHE, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


def hash_linha(linha: str) -> str:
    return hashlib.md5(linha.encode("utf-8")).hexdigest()


cache = carregar_cache()
resultados: list[dict] = []
novas, do_cache = 0, 0

for i, linha in enumerate(resenhas, start=1):
    chave = hash_linha(linha)

    if chave in cache:
        resultados.append(cache[chave])
        do_cache += 1
        print(f"  [{i}/{len(resenhas)}] do cache", end="\r")
        continue

    resultado = analisar_resenha(linha)
    if resultado is not None:
        cache[chave] = resultado
        resultados.append(resultado)
        salvar_cache(cache)  # persiste a cada nova análise
        novas += 1
        print(f"  [{i}/{len(resenhas)}] nova análise", end="\r")

print(f"\n\n✅ Concluído.")
print(f"   Novas análises: {novas}")
print(f"   Lidas do cache: {do_cache}")
print(f"   Total:          {len(resultados)}")

  [25/25] nova análise

✅ Concluído.
   Novas análises: 25
   Lidas do cache: 0
   Total:          25


## Etapa 4 — Contagens e string concatenada

Esta é a função que o desafio pede: recebe a lista de dicionários, conta positivas/negativas/neutras, e junta tudo numa string única separada por um delimitador.

Usei `Counter` da `collections` porque é mais limpo que ficar incrementando contador na mão. O separador padrão é `" | "`, mas a função aceita qualquer um.

In [ ]:
def analisar_resultados(
    lista: list[dict],
    separador: str = " | "
) -> tuple[int, int, int, str]:

    contagem = Counter(item.get("avaliacao", "").lower().strip() for item in lista)

    positivas = contagem.get("positiva", 0)
    negativas = contagem.get("negativa", 0)
    neutras = contagem.get("neutra", 0)

    partes = [json.dumps(item, ensure_ascii=False) for item in lista]
    texto_unico = separador.join(partes)

    return positivas, negativas, neutras, texto_unico


positivas, negativas, neutras, texto_concatenado = analisar_resultados(resultados)

print("=" * 50)
print("📈 RELATÓRIO")
print("=" * 50)
print(f"😊 Positivas: {positivas}")
print(f"😠 Negativas: {negativas}")
print(f"😐 Neutras:   {neutras}")
print(f"📦 Total:     {positivas + negativas + neutras}")
print("=" * 50)
print(f"\nString concatenada: {len(texto_concatenado)} caracteres")

## Etapa 5 — Visualizar as contagens

Número solto é uma coisa, gráfico é outra. Um bar chart simples deixa óbvio onde tá a maior parte das avaliações. Cores intuitivas: verde pra positivo, vermelho pra negativo, cinza pra neutro.

In [ ]:
categorias = ["Positivas", "Negativas", "Neutras"]
valores = [positivas, negativas, neutras]
cores = ["#4CAF50", "#F44336", "#9E9E9E"]

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(categorias, valores, color=cores, edgecolor="black", linewidth=0.5)

# Coloca o número em cima de cada barra
for barra, valor in zip(barras, valores):
    altura = barra.get_height()
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        altura + 0.15,
        str(valor),
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=12,
    )

ax.set_title("Distribuição das avaliações", fontsize=14, pad=15)
ax.set_ylabel("Quantidade de resenhas")
ax.set_ylim(0, max(valores) + 2)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## Etapa 6 — Análise por idioma

Como pedimos o campo `idioma` lá no prompt da Etapa 2, agora dá pra cruzar com a avaliação. Útil pra ver coisas tipo: "as reclamações se concentram em algum idioma específico?" ou "qual o público mais satisfeito?".

In [ ]:
# Quantas resenhas por idioma
idiomas = Counter(item.get("idioma", "desconhecido").lower() for item in resultados)

print("Resenhas por idioma:")
for idioma, qtd in idiomas.most_common():
    barra = "█" * qtd
    print(f"  {idioma:15s} {barra} {qtd}")

# Cruzamento idioma x sentimento
print("\n" + "-" * 50)
print("Sentimento por idioma:")
print("-" * 50)

por_idioma = defaultdict(Counter)
for item in resultados:
    idioma = item.get("idioma", "desconhecido").lower()
    avaliacao = item.get("avaliacao", "").lower()
    por_idioma[idioma][avaliacao] += 1

for idioma in sorted(por_idioma.keys()):
    s = por_idioma[idioma]
    pos = s.get("positiva", 0)
    neg = s.get("negativa", 0)
    neu = s.get("neutra", 0)
    print(f"  {idioma:15s} 😊 {pos}  😠 {neg}  😐 {neu}")

## Etapa 7 — Índice de satisfação

Aqui a gente converte os sentimentos em números (positiva = +1, neutra = 0, negativa = −1) e tira a média. O resultado é um valor entre −1 e +1 que resume o "humor geral" das resenhas num único número.

Útil pra acompanhar evolução ao longo do tempo, ou comparar versões diferentes do app.

In [ ]:
mapa = {"positiva": 1, "neutra": 0, "negativa": -1}

valores_numericos = [
    mapa.get(item.get("avaliacao", "").lower().strip(), 0)
    for item in resultados
]

indice = sum(valores_numericos) / len(valores_numericos) if valores_numericos else 0

print(f"Índice de satisfação: {indice:+.2f}  (escala de -1 a +1)")
print()

if indice >= 0.6:
    print("🌟 Usuários estão amando o app.")
elif indice >= 0.2:
    print("🙂 Saldo positivo, mas tem espaço pra melhorar.")
elif indice >= -0.2:
    print("😐 Opiniões divididas — empate técnico.")
elif indice >= -0.6:
    print("😕 Saldo negativo, hora de revisar pontos de dor.")
else:
    print("🚨 Crise de reputação — feedback majoritariamente ruim.")

## Salvando os resultados

Antes de fechar, salva tudo em um arquivo JSON. Se você rodar o notebook de novo o cache evita reprocessamento, mas ter o resultado limpinho em um arquivo separado ajuda a compartilhar ou analisar em outro lugar (Excel, Power BI, outra ferramenta).

In [ ]:
with open("resenhas_processadas.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("💾 Salvo em 'resenhas_processadas.json'")
print(f"   {len(resultados)} resenhas processadas")